# 🌳 Step 6 — Forest Density & Land-Cover Layers
**Forest Carbon Continuity Pipeline · Sentinel-1/2 Hybrid for EU LULUCF Compliance**

> ⚠️ **Architectural note:** this module produces standard LULC/density mapping (VDF/MDF/Open)
for land-cover context — but the **core carbon accounting (Step 5) deliberately excludes canopy
density**. Irish temperate forests are predominantly even-aged monocultures, so Species × Age-Class
predicts carbon stock far more accurately than canopy cover. Density layers answer "what does the
landscape look like", not "how much carbon is in it".

## What this step does
Classifies a synthetic NDVI scene into 8 land-cover classes, computes the **area of each class in
hectares**, and exports the classified raster plus a stats table. You can upload your own NDVI
GeoTIFF to classify instead.

## The Science
NDVI `=(NIR-Red)/(NIR+Red)` measures photosynthetic vigour, which tracks canopy closure closely
enough for operational density stratification: **VDF** (>0.7, very dense), **MDF** (0.4-0.7,
moderate), **Open** (0.1-0.4), **Sparse** (0-0.1). Negative NDVI typically means water; the
near-zero band is barren/built. Thresholds are adjustable — treat them as config, not physics.

In [ ]:
# ============================================================
# Environment detection & package setup (runs first, auto-detects
# Google Colab vs local Jupyter - nothing needs manual pip install)
# ============================================================
import sys, subprocess

def _ensure(pkg, import_name=None):
    import_name = import_name or pkg
    try:
        __import__(import_name)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    colab_files = None

_ensure("numpy"); _ensure("pandas"); _ensure("matplotlib")
_ensure("rasterio"); _ensure("geopandas"); _ensure("scikit-learn")
_ensure("openpyxl"); _ensure("ipywidgets")

def outpath(name):
    """Returns a working save path for the current environment."""
    return f"/content/{name}" if IN_COLAB else name

print(f"Environment detected: {'Google Colab' if IN_COLAB else 'Local Jupyter'}")

# ============================================================
# Unified cross-environment helpers:
# download buttons, regenerate buttons, upload widgets
# ============================================================
import zipfile
import ipywidgets as widgets
from IPython.display import display, FileLink, clear_output

_generated_files = {}  # friendly label -> filepath, used by the final "Download All" bundle

def register_file(label, path):
    if label:
        _generated_files[label] = path

def make_download_button(path, label=None, register_as=None):
    """Cross-environment download button. Colab: real browser download.
    Local Jupyter: clickable FileLink. Reads the file fresh at click time."""
    label = label or f"Download {path}"
    if register_as:
        register_file(register_as, path)
    btn = widgets.Button(description=label, button_style="info", icon="download")
    out = widgets.Output()

    def on_click():
        with out:
            clear_output(wait=True)
            if IN_COLAB:
                colab_files.download(path)
            else:
                display(FileLink(path))
                print(f"Click the link above to download '{path}'.")

    btn.on_click(on_click)
    display(widgets.VBox([btn, out]))

def make_regenerate_button(rerun_fn, label="🔄 Regenerate & Re-run"):
    """Button that calls rerun_fn() (fresh random seed, full re-run of this
    section) and shows the refreshed output inline below the button."""
    btn = widgets.Button(description=label, button_style="warning", icon="refresh")
    out = widgets.Output()

    def on_click():
        with out:
            clear_output(wait=True)
            rerun_fn()

    btn.on_click(on_click)
    display(widgets.VBox([btn, out]))

def make_upload_widget(on_loaded, accept=".csv,.xlsx", label="Upload replacement file"):
    """Colab: native blocking file picker via google.colab.files.upload().
    Local Jupyter: ipywidgets.FileUpload + Load button (non-blocking)."""
    out = widgets.Output()

    if IN_COLAB:
        btn = widgets.Button(description=label, button_style="primary", icon="upload")

        def on_click():
            with out:
                clear_output(wait=True)
                print("Choose a file in the dialog, or Cancel to keep current data.")
                uploaded = colab_files.upload()
                if uploaded:
                    fname = next(iter(uploaded))
                    on_loaded(fname, uploaded[fname])
                else:
                    print("No file selected - keeping current data.")

        btn.on_click(on_click)
        display(widgets.VBox([btn, out]))
    else:
        uploader = widgets.FileUpload(accept=accept, multiple=False, description="Choose file")
        load_btn = widgets.Button(description=label, button_style="primary", icon="upload")

        def on_click():
            with out:
                clear_output(wait=True)
                val = uploader.value
                if not val:
                    print("No file selected yet - choose a file above, then click the button.")
                    return
                item = list(val.values())[0] if isinstance(val, dict) else val[0]
                if isinstance(item, dict) and "metadata" in item:
                    fname = item["metadata"]["name"]     # ipywidgets 7.x shape
                else:
                    fname = item["name"]                  # ipywidgets 8.x shape
                on_loaded(fname, bytes(item["content"]))

        load_btn.on_click(on_click)
        display(widgets.VBox([uploader, load_btn, out]))

def make_download_all_button(label="⬇️ Download ALL outputs (.zip)"):
    """Zips every output registered so far and offers it as one download."""
    zip_path = outpath("outputs_bundle.zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for _label, _path in _generated_files.items():
            try:
                zf.write(_path)
            except FileNotFoundError:
                pass
    make_download_button(zip_path, label=label)

print("Helpers ready: make_download_button / make_regenerate_button / make_upload_widget")

In [ ]:
# ============================================================
# Standalone synthetic NDVI scene + LULC class definitions
# ============================================================
import numpy as np
import rasterio
from rasterio.transform import from_bounds

H, W = 120, 120
BOUNDS = (-6.4, 52.95, -6.1, 53.15)
TRANSFORM = from_bounds(*BOUNDS, W, H)
PIXEL_AREA_HA = 100.0 / 10000.0

CLASSES = {1: "VDF (>70%)", 2: "MDF (40-70%)", 3: "Open (10-40%)", 4: "Sparse (0-10%)",
           5: "Nursery", 6: "Barren", 7: "Built-up", 8: "Water"}
COLORS = ["#004d00", "#008000", "#33cc33", "#99ff99",
          "#ffff00", "#d2b48c", "#ff0000", "#006994"]

def generate_ndvi_scene(seed=3):
    """Synthetic NDVI with forest blobs, a nursery patch, a lake and a town."""
    rng = np.random.default_rng(seed)
    yy, xx = np.mgrid[0:H, 0:W]
    blob1 = np.exp(-(((xx - 35) / 18) ** 2 + ((yy - 40) / 16) ** 2))
    blob2 = 0.9 * np.exp(-(((xx - 85) / 22) ** 2 + ((yy - 85) / 20) ** 2))
    ndvi = np.clip(0.15 + 0.65 * blob1 + 0.6 * blob2 + rng.normal(0, 0.07, (H, W)), -0.4, 0.95)
    ndvi = ndvi.astype("float32")
    ndvi[40:50, 40:50] = 0.55 + rng.normal(0, 0.03, (10, 10))    # nursery rows
    ndvi[80:95, 70:90] = -0.25                                    # lake
    ndvi[15:25, 90:100] = -0.05                                   # built-up edge
    return ndvi

def classify_density(ndvi):
    """Threshold NDVI into the 8 LULC classes defined above."""
    density = np.zeros(ndvi.shape, dtype="uint8")
    density[ndvi > 0.7] = 1
    density[(ndvi >= 0.4) & (ndvi <= 0.7)] = 2
    density[(ndvi >= 0.1) & (ndvi < 0.4)] = 3
    density[(ndvi > 0.0) & (ndvi < 0.1)] = 4
    density[(ndvi <= 0.0) & (ndvi > -0.1)] = 6
    density[ndvi <= -0.1] = 7
    return density

ndvi = generate_ndvi_scene()
density = classify_density(ndvi)
density[40:50, 40:50] = 5   # nursery patch (planted rows, moderate NDVI)
density[80:95, 70:90] = 8   # water body
print(f"NDVI scene: {H}x{W} -> LULC classes {sorted(np.unique(density).tolist())}")

In [ ]:
# ============================================================
# Classify + area statistics + map
# ============================================================
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

def run_density(ndvi, tag="synthetic"):
    density = classify_density(ndvi)
    density[40:50, 40:50] = 5
    density[80:95, 70:90] = 8

    rows = []
    for k, name in CLASSES.items():
        n = int((density == k).sum())
        rows.append({"Class": k, "LULC": name, "Pixels": n,
                     "Area (ha)": round(n * PIXEL_AREA_HA, 1)})
    stats = pd.DataFrame(rows)
    stats["Share (%)"] = (stats["Pixels"] / stats["Pixels"].sum() * 100).round(1)
    print(stats.to_string(index=False))

    fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))
    cmap = ListedColormap(COLORS)
    axes[0].imshow(density, cmap=cmap, interpolation="nearest", vmin=1, vmax=8)
    axes[0].set_title("Forest density & land cover")
    handles = [Patch(facecolor=COLORS[k - 1], label=f"{k}: {v}") for k, v in CLASSES.items()]
    axes[0].legend(handles=handles, bbox_to_anchor=(1.02, 1), loc="upper left",
                   fontsize=8, title="LULC classes", frameon=False)
    axes[1].barh(stats["LULC"], stats["Area (ha)"], color=COLORS)
    axes[1].set_title("Area per class (ha)")
    axes[1].set_xlabel("Hectares")
    plt.tight_layout()

    chart_path = outpath("step6_lulc_chart.png")
    fig.savefig(chart_path, dpi=120, bbox_inches="tight")
    plt.show()

    tif_path = outpath("step6_density_layers.tif")
    with rasterio.open(tif_path, "w", driver="GTiff", height=H, width=W, count=1,
                       dtype="uint8", crs="EPSG:4326", transform=TRANSFORM) as dst:
        dst.write(density, 1)

    xlsx_path = outpath("step6_lulc_area_stats.xlsx")
    stats.to_excel(xlsx_path, index=False)

    make_download_button(chart_path, label="⬇️ Download LULC chart (.png)",
                         register_as=f"Step 6 chart ({tag})")
    make_download_button(tif_path, label="⬇️ Download classified raster (.tif)",
                         register_as=f"Step 6 raster ({tag})")
    make_download_button(xlsx_path, label="⬇️ Download area stats (.xlsx)",
                         register_as=f"Step 6 stats ({tag})")
    return stats

stats = run_density(ndvi)

**Interpretation:** the left map is the classic forestry stratification — dark green VDF cores
around the plantation blobs, lighter MDF fringes, and non-forest classes (nursery, lake, built-up)
carved out explicitly. The bar chart quantifies each class in hectares; those areas feed directly
into LULUCF "land converted to forest land" tables. And per the architectural note above: this
stratification is for **context and change tracking** — the carbon number still comes from the
Species × Age-Class matrix in Step 5, not from canopy density.

In [ ]:
# ============================================================
# Optional: upload your own NDVI GeoTIFF to classify
# ============================================================
def handle_ndvi_upload(fname, content_bytes):
    tmp = outpath("uploaded_ndvi.tif")
    open(tmp, "wb").write(content_bytes)
    try:
        with rasterio.open(tmp) as src:
            u_ndvi = src.read(1).astype("float32")
    except Exception as e:
        print(f"❌ Could not read '{fname}': {e}")
        return
    if u_ndvi.min() < -1.0 or u_ndvi.max() > 1.0:
        print(f"❌ Upload rejected - NDVI must be within [-1, 1], found "
              f"[{u_ndvi.min():.2f}, {u_ndvi.max():.2f}].")
        return
    print(f"✅ SUCCESS: loaded NDVI from '{fname}'. Re-classifying...")
    run_density(u_ndvi, tag=f"uploaded:{fname}")

make_upload_widget(handle_ndvi_upload, accept=".tif,.tiff", label="Upload NDVI (.tif)")
print("Skip to continue with the synthetic scene.")

# ============================================================
# Regenerate control
# ============================================================
def rerun_step6(seed=None):
    """Fresh synthetic NDVI scene + full classification re-run."""
    global ndvi, stats
    seed = seed or np.random.randint(0, 10_000)
    ndvi = generate_ndvi_scene(seed=seed)
    print(f"[seed={seed}] regenerated NDVI scene")
    stats = run_density(ndvi, tag=f"seed-{seed}")

make_regenerate_button(rerun_step6, label="🔄 Regenerate scene & re-classify")
make_download_all_button()